In [1]:
import sys
print(sys.executable)

c:\Users\HP\anaconda3\envs\repomind\python.exe


## Step 2: Ingestion — Reading Text Out of a PDF

Before any embeddings, vector search, or LLM calls, the pipeline needs
clean text to work with. This is the "ingestion" stage of RAG.

**Why start here instead of the exciting ML parts:**
If text extraction is messy (broken spacing, missing sections, garbled
tables), everything downstream — chunking, embeddings, retrieval — will
silently inherit that damage. Bad ingestion is the #1 hidden cause of
"why is my RAG system giving bad answers" in real systems, so we test
it in isolation first.

**What we're checking in this step:**
- Can we open the PDF and get the right number of pages?
- Does the extracted text read cleanly, or is it garbled?

We're using `pypdf` — a lightweight, pure-Python PDF library. No LLM,
no embeddings yet. Just raw text extraction.

In [2]:
from pypdf import PdfReader

reader = PdfReader("48LawsOfPower.pdf")
print(f"Number of pages: {len(reader.pages)}")

Number of pages: 476


In [3]:
first_page_text = reader.pages[10].extract_text()
print(first_page_text[:500])

LAW 14 page 107
POSE AS A FRIEND, WORK AS A SPY
Knowing about your rival is critical. Use spies to gather valuable information that will keep you a step ahead.
Better still: Play the spy yourself In polite social encounters, learn to probe. Ask indirect questions to get people
to reveal their weaknesses and intentions. There is no occasion that is not an opportunity finr artful spying.
LAW J5 page 107
CRUSH YOUR ENEMY TOTALLY
All great leaders since Adoses have known that a feared enemy must be 


## Step 4: Loading All Pages Into a Structured List

Right now we're pulling one page at a time. For chunking to work later,
we need every page's text stored together with *which page it came from*
— that page number becomes the citation later ("this answer came from
page 14"). So we build a list of dictionaries, not just raw strings.

In [4]:
pages_data = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():  # Skip blank pages and pages with only images
        pages_data.append({
            "page_number": i + 1,
            "text": text.strip()
        })

print(f"Total pages with extractable text: {len(pages_data)} out of {len(reader.pages)} ")
print(f"Example record : \n {pages_data[10]}")

Total pages with extractable text: 475 out of 476 
Example record : 
 {'page_number': 12, 'text': 'LIXW 20 page 14;)’\nDO NOT COMMIT TO ANYONE\nIt is thefool who always rushes to take sides. Do not commit to any side or cause but yourseh‘. By maintaining\nyour independence, you become the master of(ufhers—~playingj7eo[1le against one another: making them pursue\nyou.\nLAW 21 page 156\nPLAY A SUCKER TO CATCH A SUCKER—--SEEM DUMBER THAN YOUR MARK\nNo one likesjeeling smpider than the next person. The trick, then, is to make your victimsjeel smart—and not\njust smart, out smarter than you are. Once convinced o/"this, they will never suspect that you may have ulte-\nrior motives‘\nLAW 22 page 163\nUSE THE SURRENDER TACTIC: TRANSFORM WEAKNESS INTO POWER\nWhen you are weaker; neverfight for honor’s sake; choose surrender instead. Surrender gives you time to ne-\ncouer, time to torment and irritate your conqueror; time to waitfor his power to wane. Do not give him the sat»\nisfaction offighti

In [5]:
raw_sample = pages_data[10]["text"]
print(raw_sample[:400])


LIXW 20 page 14;)’
DO NOT COMMIT TO ANYONE
It is thefool who always rushes to take sides. Do not commit to any side or cause but yourseh‘. By maintaining
your independence, you become the master of(ufhers—~playingj7eo[1le against one another: making them pursue
you.
LAW 21 page 156
PLAY A SUCKER TO CATCH A SUCKER—--SEEM DUMBER THAN YOUR MARK
No one likesjeeling smpider than the next person. The tr


In [6]:
raw_sample = pages_data[10]["text"]
print(repr(raw_sample[:400]))

'LIXW 20 page 14;)’\nDO NOT COMMIT TO ANYONE\nIt is thefool who always rushes to take sides. Do not commit to any side or cause but yourseh‘. By maintaining\nyour independence, you become the master of(ufhers—~playingj7eo[1le against one another: making them pursue\nyou.\nLAW 21 page 156\nPLAY A SUCKER TO CATCH A SUCKER—--SEEM DUMBER THAN YOUR MARK\nNo one likesjeeling smpider than the next person. The tr'


In [7]:
# Basic Cleaning using Regex

import re

def clean_text(text: str) -> str:
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for p in pages_data:
    p["text"] = clean_text(p["text"])

In [8]:
print(pages_data[10]["text"][:400])

LIXW 20 page 14;)’ DO NOT COMMIT TO ANYONE It is thefool who always rushes to take sides. Do not commit to any side or cause but yourseh‘. By maintaining your independence, you become the master of(ufhers—~playingj7eo[1le against one another: making them pursue you. LAW 21 page 156 PLAY A SUCKER TO CATCH A SUCKER—--SEEM DUMBER THAN YOUR MARK No one likesjeeling smpider than the next person. The tr


## Step 5: Chunking - Splitting Pages into Retrival-Sized Pieces

We now have clean, continuous text per page. But a whole page (400-600
words) is still too coarse a unit to retrieve well — as discussed
earlier, embedding a whole page blurs together multiple unrelated laws
into one vector.

**Plan:**
1. Split each page's text into sentences (so we never cut mid-sentence)
2. Group sentences together until we hit a target chunk size (~150-200
   words)
3. Carry a small overlap (last ~30-40 words) into the next chunk, so
   content sitting near a chunk boundary isn't lost from both sides

Let's build and test this in two small steps: first just sentence
splitting (verify it looks right), then the full chunk-grouping logic.

In [9]:
def split_sentences(text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip()]

sample_sentences = split_sentences(pages_data[10]["text"])
print(f"Page {pages_data[10]['page_number']} -> {len(sample_sentences)} sentences\n")

for s in sample_sentences[:10]:
    print("-", s)

Page 12 -> 19 sentences

- LIXW 20 page 14;)’ DO NOT COMMIT TO ANYONE It is thefool who always rushes to take sides.
- Do not commit to any side or cause but yourseh‘.
- By maintaining your independence, you become the master of(ufhers—~playingj7eo[1le against one another: making them pursue you.
- LAW 21 page 156 PLAY A SUCKER TO CATCH A SUCKER—--SEEM DUMBER THAN YOUR MARK No one likesjeeling smpider than the next person.
- The trick, then, is to make your victimsjeel smart—and not just smart, out smarter than you are.
- Once convinced o/"this, they will never suspect that you may have ulte- rior motives‘ LAW 22 page 163 USE THE SURRENDER TACTIC: TRANSFORM WEAKNESS INTO POWER When you are weaker; neverfight for honor’s sake; choose surrender instead.
- Surrender gives you time to ne- couer, time to torment and irritate your conqueror; time to waitfor his power to wane.
- Do not give him the sat» isfaction offighting and defeating you——surrender first.
- By turning the other cheek you 

## Known limitation (documented, not fixed)

Page headers/footers (e.g. "LAW 20 page 14") sometimes get glued onto
the following real sentence, since they have no terminal punctuation
for our splitter to detect. This is a minor, low-impact issue — it adds
a bit of noise to a small number of chunks but does not break retrieval
or generation. A future improvement would be a regex pattern to strip
lines matching `LAW \d+ page \d+` before sentence-splitting.

In [10]:
def group_into_chunks(sentences: list[str], chunk_size_words: int = 180, overlap_words: int = 30) -> list[str]:
    chunks = []
    current_words = []

    for sentence in sentences:
        sentence_words = sentence.split()
        if len(current_words) + len(sentence_words) > chunk_size_words and current_words:
            chunks.append(" ".join(current_words))
            current_words = current_words[-overlap_words:]  # carry overlap forward
        current_words.extend(sentence_words)

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks

# test on one page first
test_chunks = group_into_chunks(sample_sentences)
print(f"{len(sample_sentences)} sentences -> {len(test_chunks)} chunk(s)\n")
for i, c in enumerate(test_chunks):
    print(f"--- chunk {i} ({len(c.split())} words) ---")
    print(c[:200], "...\n")

19 sentences -> 3 chunk(s)

--- chunk 0 (164 words) ---
LIXW 20 page 14;)’ DO NOT COMMIT TO ANYONE It is thefool who always rushes to take sides. Do not commit to any side or cause but yourseh‘. By maintaining your independence, you become the master of(uf ...

--- chunk 1 (180 words) ---
conqueror; time to waitfor his power to wane. Do not give him the sat» isfaction offighting and defeating you——surrender first. By turning the other cheek you infiiriate and unsettle him. Make surrend ...

--- chunk 2 (127 words) ---
ofcorurtiership and there will be no limit to howfaryou can rise in the court. LAW 25 page 19;/' RE-CREATE YOURSELF Do not acvept the roles that society foists on you. Rmreate yourself by forging a ne ...



## Step 6: Apply Chunking Across the Entire Document

We tested on one page. Now we run this across all 476 pages, and —
critically — we attach metadata to every chunk: which page it came
from, and its position within that page. Without this metadata, we'd
have great chunks but no way to cite where an answer came from later.

This is the final structured output ingestion produces: a flat list of
chunks, each one ready to be embedded in the next step.

In [11]:
all_chunks = []

for page in pages_data:
    sentences = split_sentences(page["text"])
    page_chunks = group_into_chunks(sentences)
    for idx, chunk_text in enumerate(page_chunks):
        all_chunks.append({
            "text": chunk_text,
            "page_number": page["page_number"],
            "chunk_index": idx,
            "chunk_id": f"page{page['page_number']}_chunk{idx}"   # unique id, useful later
        })

print(f"Total chunks across {len(pages_data)} pages: {len(all_chunks)}")
print(f"Average chunks per page: {len(all_chunks) / len(pages_data):.2f}")
print(f"\nExample chunk:\n{all_chunks[20]}")

Total chunks across 475 pages: 1822
Average chunks per page: 3.84

Example chunk:
{'text': 'ally break out. More is lost through stopping halfway than through total annihilation: The enemy will recover, and will seek revenge. Crush him, not only in body but in spirit. LAW 16 page 115 USE ABSENCE TO INCREASE RESPECT AND HONOR Too much circulation makes the price go down: The more you are seen and heardfrom, the more common you appear Ifyou are already established in a group, temporary withdrawalfrom it will make you more talked about, even more admired. You must learn when to leave. Create value through scarcity. LAW 17 page 1:23 KEEP OTHERS IN SUSPENDED TERROR: CULTIVATE AN AIR OF UNPREDICTABILITY Humans are creatures of habit with an insatiable need to seefamiliarity in other peoples actions. Your pre- dictability gives them a sense of control. Turn the tables: Be deliberately unpredictable. Behavior that seems to have no consistency orpurpose will keep them off—balance, and they will

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. Embedding dimension:", model.get_embedding_dimension())

c:\Users\HP\anaconda3\envs\repomind\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4766.73it/s]


Model loaded. Embedding dimension: 384


## Step 8: Embed a Chunk and Actually Look at the Vector

Before embedding all ~1000+ chunks (which takes a little time), let's
embed just one chunk and look at the raw output. It's easy to treat
"embedding" as a black box — seeing the actual numbers once makes it
concrete instead of magic.

In [13]:
sample_chunk = all_chunks[20]["text"]
vector = model.encode(sample_chunk)

print(f"Chunk text: {sample_chunk[:100]}...")
print(f"\nVector type: {type(vector)}")
print(f"Vector shape: {vector.shape}")
print(f"First 10 values: {vector[:10]}")

Chunk text: ally break out. More is lost through stopping halfway than through total annihilation: The enemy wil...

Vector type: <class 'numpy.ndarray'>
Vector shape: (384,)
First 10 values: [ 0.04360875  0.04746805 -0.00155319  0.02245654  0.01342602  0.02598298
  0.00766435 -0.01788325  0.04827632 -0.0406809 ]


## Step 8.5: Sanity Check — Do Similar Meanings Actually Get Similar Vectors?

Before trusting this model on our real data, let's verify the core
claim with a tiny controlled test: two sentences with similar meaning
but different wording should have vectors that are close together
(high cosine similarity), while an unrelated sentence should be
farther away.

This is a cheap, fast way to catch a broken setup before building
anything on top of it — if this test fails, nothing downstream will
work, no matter how good our chunking was.

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

test_sentences = [
    "Never let anyone know your true intentions.",      # A
    "Keep your real plans hidden from others.",          # B - similar meaning to A
    "The recipe calls for two cups of flour.",           # C - unrelated
]

test_vectors = model.encode(test_sentences)

sim_matrix = cosine_similarity(test_vectors)
print("Cosine similarity matrix:")
print(np.round(sim_matrix, 3))

Cosine similarity matrix:
[[1.    0.504 0.015]
 [0.504 1.    0.022]
 [0.015 0.022 1.   ]]


## Step 9: Batch-Embedding Every Chunk

Now we embed all ~1000+ chunks at once. We pass the whole list to
`model.encode()` in one call rather than looping chunk-by-chunk —
batching lets the model process multiple chunks in parallel internally,
which is significantly faster than one-at-a-time encoding.

We'll also time it, since this is the first step where performance
actually becomes noticeable, and it's worth knowing that going in.

In [15]:
import time

chunk_texts = [c["text"] for c in all_chunks]

start = time.time()
all_vectors = model.encode(chunk_texts, show_progress_bar=True, batch_size=32)
elapsed = time.time() - start

print(f"Embedded {len(chunk_texts)} chunks in {elapsed:.1f} seconds")
print(f"Shape of result: {all_vectors.shape}")

Batches: 100%|██████████| 57/57 [01:13<00:00,  1.29s/it]

Embedded 1822 chunks in 73.7 seconds
Shape of result: (1822, 384)


In [17]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name ="laws_of_power",
    metadata={"hsnw:space": "cosine"}
)

print("Collection:", collection.name)
print("Existing items:", collection.count())

Collection: laws_of_power
Existing items: 0


## Inserting All Chunks + Vectors into Chroma

Chroma's `.add()` method takes four parallel lists, matched by position
one final time — but this time, once inserted, they're permanently
linked together as single records inside the database. No more manual
index-matching after this point.

- `embeddings`: the 384-dim vectors (already computed)
- `documents`: the actual readable chunk text
- `metadatas`: page_number and chunk_index, for citations later
- `ids`: a unique string ID per chunk (we already built chunk_id for this)

Chroma has a per-call batch size limit, so for very large datasets you'd
insert in batches. 1822 is small enough to insert in one call.

In [19]:
collection.add(
    embeddings=all_vectors.tolist(),
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"page_number":c["page_number"], "chunk_index":c["chunk_index"] } for c in all_chunks],
    ids=[c["chunk_id"] for c in all_chunks ],
)

print("Items now in collection:", collection.count())

Items now in collection: 1822


## Query Chroma with a Real Question

To search, we do exactly what we discussed earlier:
1. Embed the question using the SAME model we used for the chunks
   (all-MiniLM-L6-v2) — mixing embedding models would produce vectors
   that aren't comparable to each other.
2. Pass that question vector to `collection.query()`, asking for the
   top-k most similar stored chunks.
3. Chroma returns the matched documents (text), their metadata
   (page_number), and a distance score (lower = more similar, for
   cosine distance).

This is the actual "R" in RAG — everything before this was setup.

In [20]:
question = "What does the book say about surrendering to a stronger enemy"

In [21]:
question_vector = model.encode(question).tolist()

results = collection.query(
    query_embeddings = [question_vector],
    n_results=3
)

for i in range(len(results["documents"][0])):
    doc = results["documents"][0][i]
    meta = results["metadatas"][0][i]
    distance=results["distances"][0][i]
    print(f"--- Match {i+1} (page {meta['page_number']}, distance {distance:.4f}) ---")
    print(doc[:300])
    print()

--- Match 1 (page 186, distance 0.6882) ---
LAW 22 USE THE SURRENDER TACTIC: TRANSFORM WEAKNESS INTO POWER J U D G M E N T When you are weaker, neverfightfor honor’: sake; choose surrender instead. Surrender gives you time to recover; time to torment and irritate your conqueror, time to wait for his power to wane. Do not give him the satisfac

--- Match 2 (page 191, distance 0.7450) ---
the blessings of perfect tranquillity and our hegemony is acknowledged throughout the globe.” This is a brilliant application of the Law: Use sur- render to gain access to your enemy. Learn his ways, insinuate yourself with him slowly, outwardly conform to his customs, but inwardly maintain your own

--- Match 3 (page 190, distance 0.7838) ---
your lack of resistance. By yielding, you in fact control the situation, because your surrender is part of a larger plan to lull them into believing they have defeated you. This is the essence of the surrender tactic: Inwardly you stay firm, but outwardly you bend

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your-api-key-here" 

In [23]:
from groq import Groq

client = Groq()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}
    ]
)

print(response.choices[0].message.content)

Retrieval-augmented generation is a natural language processing technique that combines the strengths of retrieval-based and generation-based approaches, using a retrieval component to fetch relevant information from a database or knowledge base and a generation component to create text based on that information.


In [ ]:
def ask_repomind(question: str, n_results: int = 3) -> str:
    # 1. Retrieve
    question_vector = model.encode(question).tolist()
    results = collection.query(query_embeddings=[question_vector], n_results=n_results)

    retrieved_chunks = results["documents"][0]
    retrieved_metas = results["metadatas"][0]

    # 2. Build context block, with page citations
    context_block = "\n\n".join([
        f"[Page {meta['page_number']}]: {chunk}"
        for chunk, meta in zip(retrieved_chunks, retrieved_metas)
    ])

    # 3. Construct the prompt
    prompt = f"""Answer the question using ONLY the context below. 
If the context does not contain enough information to answer, say so clearly — do not use outside knowledge.
Cite the page number(s) your answer is based on.

Context:
{context_block}

Question: {question}

Answer:"""

    # 4. Call the LLM
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# test it
answer = ask_repomind("What does the book say about surrendering to a stronger enemy?")
print(answer)

The book suggests that surrendering to a stronger enemy can be a powerful tactic, as it gives you time to recover, undermine, and potentially sabotage the enemy. By surrendering, you can lull the enemy into complacency, making them less likely to take precautions against you, and ultimately emerge victorious. This approach is considered a "soft, penetrable form of invasion" that can be more effective than outright resistance. (Pages 186, 188, 191)


: 